In [1]:
!pip install faiss-cpu open_clip_torch accelerate qwen-vl-utils evaluate rouge_score bert_score scikit-learn -q

In [2]:
import torch
import faiss
import open_clip
import time
import evaluate
import numpy as np
import gc
import random
from sklearn.metrics import accuracy_score, f1_score
from datasets import load_dataset
from PIL import Image, ImageFile
from transformers import pipeline

ImageFile.LOAD_TRUNCATED_IMAGES = True

# --- 1. ĐƯỜNG DẪN TRÊN KAGGLE ---
INDEX_PATH = "/kaggle/input/datasets/trananhvu1412/pathvqa-text-only/pathvqa_textonly.index"
META_PATH = "/kaggle/input/datasets/trananhvu1412/pathvqa-text-only/pathvqa_textonly_metadata.pt"

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


class PathVqaTextOnlyRetriever:
    def __init__(self):
        self.device = "cuda:0" if torch.cuda.is_available() else "cpu"
        
        # Tải Faiss Index và Metadata Text-only
        self.faissIndex = faiss.read_index(INDEX_PATH)
        self.metaDataList = torch.load(META_PATH, weights_only=False)
        
        # Chỉ khởi tạo Text Encoder từ BioMedCLIP
        modelName = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
        self.bioMedClip, _, _ = open_clip.create_model_and_transforms(modelName)
        self.tokenizer = open_clip.get_tokenizer(modelName)
        self.bioMedClip.to(self.device).eval()

    def retrieveTopK(self, questionText, topK=5):
        with torch.no_grad():
            tokens = self.tokenizer([questionText]).to(self.device)
            
            # Chỉ Encode Text và chuẩn hóa
            txtFeat = self.bioMedClip.encode_text(tokens)
            txtFeat /= txtFeat.norm(dim=-1, keepdim=True)

            queryVec = txtFeat.cpu().numpy().astype('float32')
            distances, indices = self.faissIndex.search(queryVec, topK)
            
            return [self.metaDataList[idx] for idx in indices[0]], distances[0]


class SlmGenerator:
    def __init__(self):
        print("\n[Hạ tầng] Đang tải Qwen2-VL-2B-Instruct (RAG K=5 Text-only)...")
        self.pipe = pipeline(
            "image-text-to-text",
            model="Qwen/Qwen2-VL-2B-Instruct",
            device_map="auto",
            model_kwargs={"torch_dtype": torch.bfloat16} 
        )
        print("Đã nạp thành công mô hình cấu hình 2B!")

    def optimizeImage(self, pilImg, maxRes=336):
        if pilImg.mode != "RGB": 
            pilImg = pilImg.convert("RGB")
        pilImg.thumbnail((maxRes, maxRes), Image.Resampling.LANCZOS)
        return pilImg

    def generate(self, testImg, testQuest, contexts):
        contentBlock = []
        
        systemInstruction = (
            "<System>\n"
            "You are an expert medical diagnostic assistant. Your task is to answer clinical questions "
            "based on an input pathology image and a set of retrieved similar cases.\n"
            "Strictly follow these rules:\n"
            "1. Analyze the provided [Image] and [Question].\n"
            "2. Evaluate the [Retrieved Evidence] from the knowledge base.\n"
            "Note that the retrieved cases might have conflicting answers; use them as reference, "
            "but prioritize the visual features of the current image.\n"
            "3. Provide a concise and definitive answer.\n"
            "</System>\n\n"
            "<User>\n"
            "[Retrieved Evidence (Top-3)]\n"
            "Below are similar past cases from the clinical database:\n"
        )
        contentBlock.append({"type": "text", "text": systemInstruction})
        
        # Đính kèm thông tin dạng văn bản (Text-only RAG)
        for i, ctx in enumerate(contexts):
            evidenceText = f"{i+1}. Case Reference: Question \"{ctx['question']}\" resulted in Answer \"{ctx['answer']}\".\n"
            contentBlock.append({"type": "text", "text": evidenceText})
            
        contentBlock.append({"type": "text", "text": "\n[Target Case Image]\n"})
        optimizedTestImg = self.optimizeImage(testImg)
        contentBlock.append({"type": "image", "image": optimizedTestImg})
        
        targetInstruction = (
            f"\nCurrent Question: {testQuest}\n\n"
            "Based on the visual evidence and the clinical context provided above, what is the correct answer for the current case?\n"
            "</User>\n\n"
            "<Assistant>\n"
            "Final Answer: "
        )
        contentBlock.append({"type": "text", "text": targetInstruction})
        
        messages = [{"role": "user", "content": contentBlock}]
        outputs = self.pipe(
            messages, 
            max_new_tokens=15, 
            generate_kwargs={"do_sample": False, "max_length": None}
        )
        
        rawResponse = outputs[0]['generated_text']
        finalText = rawResponse[-1]['content'] if isinstance(rawResponse, list) else rawResponse
        return finalText.replace("</Assistant>", "").strip()


def runKagglePathVqa2bTextOnlyRagBenchmark(sampleSize=500):
    print("1. Khởi tạo các thư viện đo lường (Metrics)...")
    bleuMetric = evaluate.load("bleu")
    rougeMetric = evaluate.load("rouge")
    bertscoreMetric = evaluate.load("bertscore")
    
    print("2. Tải các mô hình và kho dữ liệu PathVQA...")
    retriever = PathVqaTextOnlyRetriever()
    slm = SlmGenerator()
    testDS = load_dataset("flaviagiammarino/path-vqa", split="test")
    
    totalIndices = list(range(len(testDS)))
    sampledIndices = random.sample(totalIndices, min(sampleSize, len(testDS)))
    
    results = {"closed": {"ref": [], "pred": []}, "open": {"ref": [], "pred": []}}
    latencyList = []
    
    yesKeywords = ['yes', 'correct', 'true', 'positive', 'present', 'visible']
    noKeywords = ['no', 'incorrect', 'false', 'negative', 'absent', 'normal', 'none']
    
    print(f"\n🚀 BẮT ĐẦU CHẠY THỰC NGHIỆM TEXT-ONLY RAG (QWEN-2B | K=5) TRÊN {len(sampledIndices)} MẪU PATHVQA...")
    print("-" * 60)
    
    if torch.cuda.is_available(): 
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        
    startTotalTime = time.time()
    skippedCount = 0

    for progressCount, idx in enumerate(sampledIndices):
        item = testDS[idx]
        startTimeSample = time.time()
        
        try:
            # Chỉ truyền chuỗi văn bản câu hỏi vào Retriever
            contexts, _ = retriever.retrieveTopK(str(item['question']), topK=5)
            aiAnswerRaw = slm.generate(item['image'], str(item['question']), contexts)
        except Exception as e:
            print(f"⚠️ Lỗi ngầm tại mẫu {idx}: {e}")
            skippedCount += 1
            continue
            
        latencyList.append(time.time() - startTimeSample)
        
        aiAnswer = aiAnswerRaw.lower().strip()
        groundTruth = str(item['answer']).lower().strip()
        
        if groundTruth in ['yes', 'no']:
            results["closed"]["ref"].append(groundTruth)
            hasYes = any(word in aiAnswer for word in yesKeywords)
            hasNo = any(word in aiAnswer for word in noKeywords)
            
            if hasYes and not hasNo:
                predClean = 'yes'
            elif hasNo and not hasYes:
                predClean = 'no'
            else:
                firstYesPos = aiAnswer.find('yes') if 'yes' in aiAnswer else 9999
                firstNoPos = aiAnswer.find('no') if 'no' in aiAnswer else 9999
                predClean = 'yes' if firstYesPos < firstNoPos else ('no' if firstNoPos < firstYesPos else 'unknown')
            results["closed"]["pred"].append(predClean)
        else:
            results["open"]["ref"].append(groundTruth)
            results["open"]["pred"].append(aiAnswer)
            
        if (progressCount + 1) % 50 == 0: 
            print(f"Tiến độ Kaggle (2B - Text-Only RAG K=5): {progressCount + 1}/{len(sampledIndices)} mẫu...")

    endTotalTime = time.time()
    
    print("\n" + "="*60 + "\n📊 BÁO CÁO THỰC NGHIỆM - TEXT-ONLY RAG (QWEN-2B ON PATHVQA | K=5)\n" + "="*60)
    print(f"📌 Đã quét: {len(sampledIndices)} | Bỏ qua do lỗi: {skippedCount}")
    
    if results["closed"]["ref"]:
        accScore = accuracy_score(results["closed"]["ref"], results["closed"]["pred"])
        f1Weighted = f1_score(results["closed"]["ref"], results["closed"]["pred"], average='weighted', zero_division=0)
        print(f"[1. Closed-Ended Classification (Total: {len(results['closed']['ref'])} samples)]")
        print(f"  - Accuracy : {accScore:.4f}")
        print(f"  - F1-Score : {f1Weighted:.4f}")
    
    if results["open"]["ref"]:
        print(f"\n[2. Open-Ended Text Generation (Total: {len(results['open']['ref'])} samples)]")
        predsList = results["open"]["pred"]
        refsList = [[r] for r in results["open"]["ref"]]
        
        bleu1 = bleuMetric.compute(predictions=predsList, references=refsList, max_order=1, smooth=True)['bleu']
        bleu2 = bleuMetric.compute(predictions=predsList, references=refsList, max_order=2, smooth=True)['bleu']
        bleu3 = bleuMetric.compute(predictions=predsList, references=refsList, max_order=3, smooth=True)['bleu']
        rougeL = rougeMetric.compute(predictions=predsList, references=results["open"]["ref"])['rougeL']
        
        bertResults = bertscoreMetric.compute(predictions=predsList, references=results["open"]["ref"], lang="en")
        print(f"  - BLEU-1   : {bleu1:.4f}")
        print(f"  - BLEU-2   : {bleu2:.4f}")
        print(f"  - BLEU-3   : {bleu3:.4f}")
        print(f"  - ROUGE-L  : {rougeL:.4f}")
        print(f"  - BERTScore: {np.mean(bertResults['f1']):.4f}")

    totalDuration = endTotalTime - startTotalTime
    if latencyList:
        print("\n[3. Computational Efficiency Measurements]")
        print(f"  - Inference Latency (Độ trễ trung bình) : {np.mean(latencyList):.2f} s/mẫu")
        print(f"  - Throughput (Tốc độ xử lý)             : {len(latencyList) / totalDuration:.2f} mẫu/giây")
    
    if torch.cuda.is_available():
        print("\n[4. Multi-GPU Memory Profile]")
        for gpuId in range(torch.cuda.device_count()):
            vramMB = torch.cuda.max_memory_allocated(device=gpuId) / (1024 ** 2)
            print(f"  - Peak Memory Usage on GPU {gpuId}      : {vramMB:.2f} MB")
    print("="*60)
    
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

if __name__ == "__main__":
    runKagglePathVqa2bTextOnlyRagBenchmark(sampleSize=500)

1. Khởi tạo các thư viện đo lường (Metrics)...


2. Tải các mô hình và kho dữ liệu PathVQA...


open_clip_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

open_clip_pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  784MB            

open_clip_pytorch_model.bin: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

... (56 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  - BLEU-1   : 0.0348
  - BLEU-2   : 0.0191
  - BLEU-3   : 0.0119
  - ROUGE-L  : 0.0583
  - BERTScore: 0.8392

[3. Computational Efficiency Measurements]
  - Inference Latency (Độ trễ trung bình) : 1.19 s/mẫu
  - Throughput (Tốc độ xử lý)             : 0.83 mẫu/giây

[4. Multi-GPU Memory Profile]
  - Peak Memory Usage on GPU 0      : 3910.76 MB
  - Peak Memory Usage on GPU 1      : 2213.99 MB
